# KI-1: Natural Token Experiment

## 실험 개요

**연구 질문:** Mutual Cognition(상호 인지)이 에이전트 간 통신에서 토큰 사용량을 줄이면서도 정확도를 유지하거나 개선하는가?

### 에이전트 구성
- **Agent A (Transmitter):** GPT-4o - 대규모 데이터셋을 받아 분석/요약하여 전달
- **Agent B (Receiver):** GPT-4o - A의 출력에서 특정 수치를 추출하여 계산

### 실험 조건 (4가지)
| 조건 | 설명 |
|------|------|
| **No Context** | A는 청중을 모른 채 전체 데이터를 종합 보고서로 작성 |
| **One-Way** | A는 B의 전문 분야만 알고 관련 데이터를 강조 |
| **Mutual** | B가 먼저 필요한 데이터를 명시 -> A는 요청된 것만 전송 |
| **Progressive** | 3라운드에 걸쳐 A가 점진적으로 초점을 좁힘 |

### 데이터 설계
- **15개 문제**, 각각 15-20개 데이터 포인트를 포함하는 대규모 데이터셋
- B는 전체 중 2-5개 값만 필요하여 수치 계산 수행
- `max_tokens` 제한 없음 (A가 자유롭게 생성)

### 기대 결과
- Mutual 조건에서 토큰 사용량이 가장 적고, 정확도는 최고
- No Context에서 토큰이 가장 많고, 불필요 정보로 인해 정확도 저하 가능
- Progressive는 라운드가 진행될수록 토큰 감소 + 정확도 상승

In [ ]:
# ============================================================
# Setup: API Key and Dependencies
# ============================================================
import os
import json
import time
import re
from datetime import datetime

import requests
import pandas as pd

# Load API key from environment variable or prompt for input
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    OPENAI_API_KEY = input("OpenAI API Key를 입력하세요: ").strip()

assert OPENAI_API_KEY, "API Key가 설정되지 않았습니다."

MODEL = "gpt-4o"
TEMPERATURE = 0
API_URL = "https://api.openai.com/v1/chat/completions"

print(f"Model: {MODEL}")
print(f"Temperature: {TEMPERATURE}")
print("API Key: OK (loaded)")

In [ ]:
# ============================================================
# Common Functions: API Call, Number Extraction, Grading
# ============================================================

def call_gpt(system_prompt: str, user_prompt: str, retries: int = 3) -> dict:
    """Call OpenAI API with retry logic. No max_tokens limit."""
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {OPENAI_API_KEY}",
    }
    payload = {
        "model": MODEL,
        "temperature": TEMPERATURE,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        # NO max_tokens — let it generate freely
    }
    for attempt in range(retries):
        try:
            resp = requests.post(API_URL, headers=headers, json=payload, timeout=120)
            if resp.status_code == 429:
                wait = (attempt + 1) * 15
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            data = resp.json()
            usage = data["usage"]
            return {
                "content": data["choices"][0]["message"]["content"],
                "tokens": usage["completion_tokens"],
                "prompt_tokens": usage["prompt_tokens"],
                "total_tokens": usage["total_tokens"],
            }
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"  Retry {attempt + 1}: {e}")
            time.sleep(5)


def extract_number(text: str):
    """Extract a numeric answer from LLM response text."""
    if "-999" in text:
        return -999
    cleaned = text.replace(",", "").replace("$", "").replace("%", "")
    # Try structured patterns first, then fallback to any number
    patterns = [
        r"(?:=|is|equals|result|answer|ratio|rate|return|revenue|margin|range|value)[:\s]*([+-]?\d+\.?\d*(?:e[+-]?\d+)?)",
        r"\*\*([+-]?\d+\.?\d*(?:e[+-]?\d+)?)\*\*",
        r"([+-]?\d+\.?\d*(?:e[+-]?\d+)?)",
    ]
    for pat in patterns:
        matches = re.findall(pat, cleaned, re.IGNORECASE)
        candidates = [float(m) for m in matches if float(m) != 0 and float(m) != -999]
        if candidates:
            return candidates[-1]  # last significant number = final answer
    return None


def check_accuracy(extracted, expected: float, tolerance: float = 0.10) -> bool:
    """Check if extracted answer is within tolerance of expected."""
    if extracted is None or extracted == -999:
        return False
    if expected == 0:
        return abs(extracted) < 0.01
    rel_error = abs(extracted - expected) / abs(expected)
    return rel_error <= tolerance


print("Common functions loaded.")

## 문제 목록

15개 문제는 각각 다른 도메인의 대규모 데이터셋 (15-20개 항목)을 포함합니다.
Agent B는 이 중 2-5개의 특정 값만 사용하여 수치 계산을 수행합니다.

| ID | Domain | B의 계산 | 필요 데이터 수 |
|----|--------|---------|---------------|
| 1 | Corporate Finance | Debt-to-Equity Ratio | 2 |
| 2 | Manufacturing | Good Units Per Day | 5 |
| 3 | Healthcare | Bed Turnover Rate | 2 |
| 4 | Investment Portfolio | Portfolio Expected Return | 5 (x2) |
| 5 | City Demographics | People Over 65 | 2 |
| 6 | Logistics | Daily Shipping Revenue | 2 |
| 7 | Energy Utility | Reserve Margin | 2 |
| 8 | Agriculture | Gross Corn Revenue | 3 |
| 9 | Retail Chain | Revenue Per Store | 2 |
| 10 | University | Student-to-Faculty Ratio | 2 |
| 11 | Sports Team | Attendance Rate | 2 |
| 12 | Weather Station | Temperature Range | 2 |
| 13 | Telecom Provider | Monthly Revenue from ARPU | 2 |
| 14 | Airline | Revenue Per Passenger | 2 |
| 15 | Real Estate | Sell-Through Rate | 2 |

In [ ]:
# ============================================================
# Problem Definitions (15 problems, each with large dataset)
# ============================================================

PROBLEMS = [
    {
        "id": 1, "domain": "Corporate Finance",
        "dataset": "Revenue=$12M, COGS=$7.2M, SGA=$1.8M, RnD=$600K, Interest=$400K, Tax_Rate=28%, Depreciation=$500K, Amortization=$200K, Total_Assets=$25M, Current_Assets=$8M, Fixed_Assets=$15M, Intangibles=$2M, Current_Liabilities=$5M, LongTerm_Debt=$8M, Equity=$12M, Shares_Outstanding=2M, Dividends=$600K, CapEx=$1.5M, Working_Capital_Change=$300K, Cash=$1.5M",
        "data_count": 20,
        "b_task": "Compute the Debt-to-Equity Ratio = LongTerm_Debt / Equity",
        "b_needs": "LongTerm_Debt=$8M, Equity=$12M",
        "formula": "LongTerm_Debt / Equity",
        "answer": 0.667, "tolerance": 0.1,
        "general_area": "financial ratios and leverage analysis",
        "b_request": "I need LongTerm_Debt and Equity values to compute Debt-to-Equity Ratio = LongTerm_Debt / Equity.",
    },
    {
        "id": 2, "domain": "Manufacturing",
        "dataset": "Production_Rate=500units/hr, Defect_Rate=2.5%, Rework_Rate=1.5%, Scrap_Rate=1.0%, Material_Cost=$15/unit, Labor_Cost=$8/unit, Overhead=$3/unit, Energy=$2/unit, Maintenance=$50K/month, Downtime=5%, Shift_Hours=8, Shifts_Per_Day=3, Workers=120, Machines=25, Capacity_Utilization=85%, Safety_Incidents=3/yr, Waste_Rate=4%, Quality_Score=94",
        "data_count": 18,
        "b_task": "Compute Good Units Per Day = Production_Rate * (1 - Defect_Rate) * Shift_Hours * Shifts_Per_Day * (1 - Downtime)",
        "b_needs": "Production_Rate=500, Defect_Rate=2.5%, Shift_Hours=8, Shifts_Per_Day=3, Downtime=5%",
        "formula": "500 * 0.975 * 8 * 3 * 0.95",
        "answer": 11115, "tolerance": 0.10,
        "general_area": "production output and efficiency metrics",
        "b_request": "I need Production_Rate, Defect_Rate, Shift_Hours, Shifts_Per_Day, and Downtime to compute Good Units Per Day.",
    },
    {
        "id": 3, "domain": "Healthcare",
        "dataset": "Beds=500, Occupancy=78%, Avg_Stay=4.2days, Admissions=3400/month, ER_Visits=8500/month, Surgery=600/month, Outpatient=12000/month, Staff_Doctors=200, Nurses=600, Admin=150, Budget=$50M/month, Revenue=$55M/month, Insurance_Claims=$45M, Patient_Satisfaction=4.2/5, Readmission_Rate=8%, Mortality_Rate=1.2%, Infection_Rate=0.5%, Wait_Time_ER=45min",
        "data_count": 18,
        "b_task": "Compute Bed Turnover Rate = Admissions / Beds",
        "b_needs": "Admissions=3400, Beds=500",
        "formula": "3400 / 500",
        "answer": 6.8, "tolerance": 0.10,
        "general_area": "hospital capacity and utilization metrics",
        "b_request": "I need Admissions per month and total Beds to compute Bed Turnover Rate = Admissions / Beds.",
    },
    {
        "id": 4, "domain": "Investment Portfolio",
        "dataset": "Stock_A_Return=12%, Stock_A_Weight=30%, Stock_A_Std=25%, Stock_B_Return=8%, Stock_B_Weight=25%, Stock_B_Std=15%, Bond_C_Return=4%, Bond_C_Weight=20%, Bond_C_Std=5%, REIT_D_Return=9%, REIT_D_Weight=15%, REIT_D_Std=18%, Cash_Return=2%, Cash_Weight=10%, Cash_Std=0%, Corr_AB=0.6, Corr_AC=-0.2, Corr_AD=0.4, Risk_Free=3%, Benchmark_Return=10%",
        "data_count": 20,
        "b_task": "Compute Portfolio Expected Return = sum of (Weight_i * Return_i) for all 5 assets",
        "b_needs": "Stock_A: 30%*12%, Stock_B: 25%*8%, Bond_C: 20%*4%, REIT_D: 15%*9%, Cash: 10%*2%",
        "formula": "0.30*12 + 0.25*8 + 0.20*4 + 0.15*9 + 0.10*2",
        "answer": 7.95, "tolerance": 0.10,
        "general_area": "portfolio return and risk analysis",
        "b_request": "I need the weight and return for each asset (Stock_A, Stock_B, Bond_C, REIT_D, Cash) to compute Portfolio Expected Return.",
    },
    {
        "id": 5, "domain": "City Demographics",
        "dataset": "Population=850000, Area=350km2, Density=2429/km2, Growth_Rate=1.8%, Median_Age=34, Under18=22%, Over65=14%, Median_Income=$52000, Unemployment=5.5%, Poverty_Rate=12%, College_Educated=38%, Homeownership=55%, Avg_Rent=$1200, Crime_Rate=4.2/1000, Schools=120, Hospitals=8, Parks=45, Public_Transit_Ridership=150000/day",
        "data_count": 18,
        "b_task": "Compute Number of people over 65 = Population * Over65_Pct",
        "b_needs": "Population=850000, Over65=14%",
        "formula": "850000 * 0.14",
        "answer": 119000, "tolerance": 0.10,
        "general_area": "population age distribution and demographics",
        "b_request": "I need Population and Over65 percentage to compute Number of people over 65.",
    },
    {
        "id": 6, "domain": "Logistics",
        "dataset": "Warehouses=12, Total_Inventory=500000units, Avg_Order_Size=150units, Orders_Per_Day=2200, Fulfillment_Rate=96%, Return_Rate=3.5%, Shipping_Cost_Avg=$8.50/order, Avg_Delivery_Days=3.2, Fleet_Size=85trucks, Truck_Capacity=500units, Fuel_Cost=$0.45/mile, Avg_Route_Miles=120, Drivers=95, Loading_Time=45min, Sorting_Errors=0.8%, Packaging_Cost=$1.20/unit, Cross_Dock_Pct=35%, Last_Mile_Cost=$4.50",
        "data_count": 18,
        "b_task": "Compute Daily Shipping Revenue Needed = Orders_Per_Day * Shipping_Cost_Avg",
        "b_needs": "Orders_Per_Day=2200, Shipping_Cost_Avg=$8.50",
        "formula": "2200 * 8.50",
        "answer": 18700, "tolerance": 0.10,
        "general_area": "order fulfillment and shipping cost analysis",
        "b_request": "I need Orders_Per_Day and Shipping_Cost_Avg to compute Daily Shipping Revenue Needed.",
    },
    {
        "id": 7, "domain": "Energy Utility",
        "dataset": "Total_Capacity=5000MW, Solar=800MW, Wind=600MW, Natural_Gas=2200MW, Nuclear=1000MW, Hydro=400MW, Peak_Demand=4200MW, Avg_Demand=3100MW, Transmission_Loss=6%, Grid_Length=12000km, Customers=1800000, Residential=1500000, Commercial=250000, Industrial=50000, Avg_Rate=$0.12/kWh, Revenue=$2.8B/yr, Maintenance=$400M/yr, Fuel_Cost=$800M/yr, Renewable_Pct=36%, Carbon_Emissions=15M_tons/yr",
        "data_count": 19,
        "b_task": "Compute Reserve Margin = (Total_Capacity - Peak_Demand) / Peak_Demand * 100%",
        "b_needs": "Total_Capacity=5000MW, Peak_Demand=4200MW",
        "formula": "(5000 - 4200) / 4200 * 100",
        "answer": 19.05, "tolerance": 0.10,
        "general_area": "grid capacity and reliability planning",
        "b_request": "I need Total_Capacity and Peak_Demand to compute Reserve Margin.",
    },
    {
        "id": 8, "domain": "Agriculture",
        "dataset": "Total_Acreage=2500acres, Corn=800acres, Wheat=600acres, Soy=500acres, Cotton=400acres, Fallow=200acres, Corn_Yield=180bu/acre, Wheat_Yield=55bu/acre, Soy_Yield=50bu/acre, Cotton_Yield=900lb/acre, Corn_Price=$5.80/bu, Wheat_Price=$7.20/bu, Soy_Price=$13.50/bu, Cotton_Price=$0.85/lb, Fertilizer_Cost=$120/acre, Irrigation_Cost=$80/acre, Labor=$200K/yr, Equipment=$150K/yr, Crop_Insurance=$45K/yr, Soil_pH=6.5",
        "data_count": 20,
        "b_task": "Compute Gross Corn Revenue = Corn_Acreage * Corn_Yield * Corn_Price",
        "b_needs": "Corn=800acres, Corn_Yield=180bu/acre, Corn_Price=$5.80/bu",
        "formula": "800 * 180 * 5.80",
        "answer": 835200, "tolerance": 0.10,
        "general_area": "crop yield and revenue projections",
        "b_request": "I need Corn acreage, Corn_Yield (bu/acre), and Corn_Price ($/bu) to compute Gross Corn Revenue.",
    },
    {
        "id": 9, "domain": "Retail Chain",
        "dataset": "Stores=450, Avg_Store_Size=25000sqft, Total_Employees=18000, Revenue=$4.2B, COGS=$2.8B, Gross_Margin=33.3%, Inventory_Turnover=8.5, Avg_Transaction=$45, Transactions_Per_Day=12000, Loyalty_Members=5M, Online_Revenue=$800M, Online_Pct=19%, Shrinkage=1.5%, Rent_Avg=$25/sqft/yr, Marketing=$200M/yr, Same_Store_Growth=3.2%, Customer_Satisfaction=4.1/5, Foot_Traffic=2M/week",
        "data_count": 18,
        "b_task": "Compute Revenue Per Store = Total_Revenue / Stores",
        "b_needs": "Revenue=$4.2B, Stores=450",
        "formula": "4200000000 / 450",
        "answer": 9333333, "tolerance": 0.10,
        "general_area": "store performance and sales metrics",
        "b_request": "I need Total Revenue and number of Stores to compute Revenue Per Store.",
    },
    {
        "id": 10, "domain": "University",
        "dataset": "Students=35000, Undergrad=25000, Grad=8000, PhD=2000, Faculty=2200, Staff=3500, Tuition_Undergrad=$42000/yr, Tuition_Grad=$38000/yr, Acceptance_Rate=22%, Yield_Rate=45%, Retention_Rate=94%, Graduation_Rate_4yr=82%, Endowment=$8.5B, Research_Funding=$600M, Dorms=8000beds, Dining_Plans=12000, Athletics_Budget=$120M, Library_Books=4M, Alumni=250000",
        "data_count": 19,
        "b_task": "Compute Student-to-Faculty Ratio = Total_Students / Faculty",
        "b_needs": "Students=35000, Faculty=2200",
        "formula": "35000 / 2200",
        "answer": 15.91, "tolerance": 0.10,
        "general_area": "enrollment and academic resource allocation",
        "b_request": "I need Total Students and Faculty count to compute Student-to-Faculty Ratio.",
    },
    {
        "id": 11, "domain": "Professional Sports Team",
        "dataset": "Roster=53players, Salary_Cap=$225M, Total_Payroll=$218M, Avg_Salary=$4.1M, Highest_Salary=$35M, Revenue=$550M, Ticket_Revenue=$180M, Broadcast=$200M, Merchandise=$80M, Sponsorship=$60M, Stadium_Capacity=72000, Avg_Attendance=68500, Season_Tickets=55000, Win_Pct=0.625, Playoffs_Made=8of10, Coaching_Staff=25, Training_Staff=15, Scouting_Budget=$12M, Draft_Picks=7",
        "data_count": 19,
        "b_task": "Compute Attendance Rate = Avg_Attendance / Stadium_Capacity * 100%",
        "b_needs": "Avg_Attendance=68500, Stadium_Capacity=72000",
        "formula": "68500 / 72000 * 100",
        "answer": 95.14, "tolerance": 0.10,
        "general_area": "team revenue and fan engagement metrics",
        "b_request": "I need Avg_Attendance and Stadium_Capacity to compute Attendance Rate.",
    },
    {
        "id": 12, "domain": "Weather Station",
        "dataset": "Avg_Temp=72F, Max_Temp=98F, Min_Temp=45F, Humidity=65%, Pressure=1013hPa, Wind_Speed=12mph, Wind_Gusts=35mph, Precipitation=4.2in/month, Rainy_Days=11/month, Sunshine_Hours=240/month, UV_Index=7, Dew_Point=58F, Cloud_Cover=40%, Visibility=10mi, Pollen_Count=8.5, Air_Quality_Index=42, Snow_Days=0, Fog_Days=3, Storm_Days=2",
        "data_count": 19,
        "b_task": "Compute Temperature Range = Max_Temp - Min_Temp",
        "b_needs": "Max_Temp=98F, Min_Temp=45F",
        "formula": "98 - 45",
        "answer": 53, "tolerance": 0.10,
        "general_area": "temperature patterns and climate analysis",
        "b_request": "I need Max_Temp and Min_Temp to compute Temperature Range.",
    },
    {
        "id": 13, "domain": "Telecom Provider",
        "dataset": "Subscribers=25M, Prepaid=10M, Postpaid=15M, ARPU=$55, Churn_Rate=1.8%/month, Network_Towers=45000, Coverage_Area=95%, 5G_Towers=12000, 5G_Coverage=55%, Data_Usage_Avg=15GB/month, Voice_Minutes_Avg=300/month, SMS_Avg=50/month, Revenue=$16.5B, Network_Investment=$3B, Customer_Acquisition_Cost=$350, Customer_Service_Calls=2M/month, Satisfaction=3.8/5, Roaming_Revenue=$400M",
        "data_count": 18,
        "b_task": "Compute Monthly Revenue from ARPU = Subscribers * ARPU",
        "b_needs": "Subscribers=25M, ARPU=$55",
        "formula": "25000000 * 55",
        "answer": 1375000000, "tolerance": 0.10,
        "general_area": "subscriber revenue and network utilization",
        "b_request": "I need total Subscribers and ARPU to compute Monthly Revenue from ARPU.",
    },
    {
        "id": 14, "domain": "Airline",
        "dataset": "Fleet=320aircraft, Routes=500, Daily_Flights=2800, Passengers_Annual=95M, Load_Factor=84%, Avg_Fare=$285, Revenue=$28B, Fuel_Cost=$8B, Labor_Cost=$7B, Maintenance=$2.5B, Leasing=$3B, Avg_Flight_Distance=1200mi, On_Time=78%, Baggage_Mishandled=2.5/1000, Hubs=5, Lounges=12, Loyalty_Members=45M, Int_Revenue_Pct=40%, Cargo_Revenue=$1.2B",
        "data_count": 19,
        "b_task": "Compute Revenue Per Passenger = Annual_Revenue / Annual_Passengers",
        "b_needs": "Revenue=$28B, Passengers_Annual=95M",
        "formula": "28000000000 / 95000000",
        "answer": 294.74, "tolerance": 0.10,
        "general_area": "passenger yield and route profitability",
        "b_request": "I need Annual Revenue and Annual Passengers to compute Revenue Per Passenger.",
    },
    {
        "id": 15, "domain": "Real Estate Developer",
        "dataset": "Projects_Active=18, Total_Units=4500, Sold_Units=3200, Avg_Price=$450K, Revenue_YTD=$1.44B, Land_Cost=$200M, Construction_Cost=$800M, Marketing=$50M, Permits=$25M, Avg_SqFt=1800, Cost_Per_SqFt=$165, Completion_Rate=72%, Avg_Days_On_Market=45, Mortgage_Rate=6.5%, Commercial_SqFt=500000, Occupancy_Commercial=88%, Rental_Income=$35M/yr, ROI_Avg=18%, Debt=$600M, Equity=$400M",
        "data_count": 20,
        "b_task": "Compute Sell-Through Rate = Sold_Units / Total_Units * 100%",
        "b_needs": "Sold_Units=3200, Total_Units=4500",
        "formula": "3200 / 4500 * 100",
        "answer": 71.11, "tolerance": 0.10,
        "general_area": "sales performance and project completion",
        "b_request": "I need Sold_Units and Total_Units to compute Sell-Through Rate.",
    },
]

print(f"Loaded {len(PROBLEMS)} problems.")
for p in PROBLEMS:
    print(f"  P{p['id']:2d} | {p['domain']:25s} | {p['data_count']} data points | answer={p['answer']}")

## Condition 1: No Context

Agent A는 수신자가 누구인지 전혀 모르는 상태에서 전체 데이터에 대한 종합 보고서를 작성합니다.
Agent B는 이 보고서에서 필요한 값을 찾아 계산합니다.

In [ ]:
# ============================================================
# Condition 1: No Context
# A writes a comprehensive report without knowing the audience.
# B extracts needed values and computes.
# ============================================================

def run_no_context(problem):
    a_system = (f"You are a data analyst. You received a comprehensive dataset about a {problem['domain']} "
                f"operation. Write a thorough analytical report covering all key metrics, trends, and "
                f"notable findings. Be comprehensive and leave nothing out. Present your analysis in a structured format.")
    a_user = f"Here is the complete dataset:\n{problem['dataset']}\n\nPlease provide a comprehensive analytical report covering all metrics."
    a_result = call_gpt(a_system, a_user)

    b_system = ("You are a specialist calculator. Using ONLY the information provided in the report below, "
                "compute the requested value. If the exact data needed is not explicitly stated in the report, "
                "output -999. Show your calculation and give the final numeric answer.")
    b_user = (f"REPORT:\n{a_result['content']}\n\nTASK: {problem['b_task']}\n\n"
              f"Compute the answer using ONLY data from the report above. If needed data is missing, output -999.")
    b_result = call_gpt(b_system, b_user)

    extracted = extract_number(b_result["content"])
    correct = check_accuracy(extracted, problem["answer"], problem["tolerance"])
    return {"a_tokens": a_result["tokens"], "b_answer": extracted, "correct": correct,
            "a_content": a_result["content"], "b_content": b_result["content"]}

print("Running Condition 1: No Context...")
results_nc = []
for p in PROBLEMS:
    print(f"  [NC] Problem {p['id']}: {p['domain']}")
    result = run_no_context(p)
    results_nc.append(result)
    status = "OK" if result["correct"] else "FAIL"
    print(f"        -> A tokens: {result['a_tokens']}, B answer: {result['b_answer']}, {status}")
    time.sleep(1)  # brief pause to avoid rate limits

nc_acc = sum(1 for r in results_nc if r["correct"]) / len(results_nc) * 100
nc_avg_tok = sum(r["a_tokens"] for r in results_nc) / len(results_nc)
print(f"\nNo Context: Avg A tokens = {nc_avg_tok:.0f}, Accuracy = {nc_acc:.1f}%")

## Condition 2: One-Way

Agent A는 B의 전문 분야(general area)를 알고, 해당 분야에 관련된 데이터를 강조한 요약을 작성합니다.
B의 구체적 요청은 알지 못합니다.

In [ ]:
# ============================================================
# Condition 2: One-Way
# A knows B's general area of interest but not the exact request.
# ============================================================

def run_one_way(problem):
    a_system = (f"You are a data analyst. The recipient is a {problem['domain']} specialist who needs data for "
                f"{problem['general_area']}. Highlight the data most relevant to their work. "
                f"Be efficient -- focus on what matters for their analysis, skip irrelevant details.")
    a_user = f"Here is the complete dataset:\n{problem['dataset']}\n\nPrepare a focused summary for the specialist."
    a_result = call_gpt(a_system, a_user)

    b_system = ("You are a specialist calculator. Using ONLY the information provided in the summary below, "
                "compute the requested value. If the exact data needed is not explicitly stated, output -999. "
                "Show your calculation and give the final numeric answer.")
    b_user = (f"SUMMARY:\n{a_result['content']}\n\nTASK: {problem['b_task']}\n\n"
              f"Compute the answer using ONLY data from the summary above. If needed data is missing, output -999.")
    b_result = call_gpt(b_system, b_user)

    extracted = extract_number(b_result["content"])
    correct = check_accuracy(extracted, problem["answer"], problem["tolerance"])
    return {"a_tokens": a_result["tokens"], "b_answer": extracted, "correct": correct,
            "a_content": a_result["content"], "b_content": b_result["content"]}

print("Running Condition 2: One-Way...")
results_ow = []
for p in PROBLEMS:
    print(f"  [OW] Problem {p['id']}: {p['domain']}")
    result = run_one_way(p)
    results_ow.append(result)
    status = "OK" if result["correct"] else "FAIL"
    print(f"        -> A tokens: {result['a_tokens']}, B answer: {result['b_answer']}, {status}")
    time.sleep(1)

ow_acc = sum(1 for r in results_ow if r["correct"]) / len(results_ow) * 100
ow_avg_tok = sum(r["a_tokens"] for r in results_ow) / len(results_ow)
print(f"\nOne-Way: Avg A tokens = {ow_avg_tok:.0f}, Accuracy = {ow_acc:.1f}%")

## Condition 3: Mutual

Agent B가 먼저 자신이 필요한 데이터를 정확히 명시합니다.
Agent A는 요청받은 데이터만 key=value 형태로 전송합니다.

In [ ]:
# ============================================================
# Condition 3: Mutual
# B tells A exactly what it needs. A sends only that.
# ============================================================

def run_mutual(problem):
    a_system = ("You are a data analyst. The recipient has requested specific data. "
                "Send ONLY what they asked for, formatted as key=value pairs. "
                "Nothing else. No explanations, no analysis, no extra data.")
    a_user = (f"Dataset:\n{problem['dataset']}\n\n"
              f"Recipient's request: \"{problem['b_request']}\"\n\n"
              f"Send only the requested data as key=value pairs.")
    a_result = call_gpt(a_system, a_user)

    b_system = ("You are a specialist calculator. Using ONLY the information provided below, "
                "compute the requested value. If the exact data needed is not explicitly stated, "
                "output -999. Show your calculation and give the final numeric answer.")
    b_user = f"DATA:\n{a_result['content']}\n\nTASK: {problem['b_task']}\n\nCompute the answer."
    b_result = call_gpt(b_system, b_user)

    extracted = extract_number(b_result["content"])
    correct = check_accuracy(extracted, problem["answer"], problem["tolerance"])
    return {"a_tokens": a_result["tokens"], "b_answer": extracted, "correct": correct,
            "a_content": a_result["content"], "b_content": b_result["content"]}

print("Running Condition 3: Mutual...")
results_mu = []
for p in PROBLEMS:
    print(f"  [MU] Problem {p['id']}: {p['domain']}")
    result = run_mutual(p)
    results_mu.append(result)
    status = "OK" if result["correct"] else "FAIL"
    print(f"        -> A tokens: {result['a_tokens']}, B answer: {result['b_answer']}, {status}")
    time.sleep(1)

mu_acc = sum(1 for r in results_mu if r["correct"]) / len(results_mu) * 100
mu_avg_tok = sum(r["a_tokens"] for r in results_mu) / len(results_mu)
print(f"\nMutual: Avg A tokens = {mu_avg_tok:.0f}, Accuracy = {mu_acc:.1f}%")

## Condition 4: Progressive (3 Rounds)

3라운드에 걸쳐 A가 점진적으로 초점을 좁힙니다:
- **Round 1:** No Context와 동일 (전체 보고서)
- **Round 2:** B의 Round 1 응답을 피드백으로 받아 더 집중된 요약 작성
- **Round 3:** B의 Round 2 응답을 피드백으로 받아 필수 데이터만 전송

In [ ]:
# ============================================================
# Condition 4: Progressive (3 rounds)
# A narrows focus over 3 rounds based on B's feedback.
# ============================================================

def run_progressive_round(problem, round_num, prev_b_content):
    """Run one round of the progressive condition."""
    if round_num == 1:
        a_system = (f"You are a data analyst. You received a comprehensive dataset about a {problem['domain']} "
                    f"operation. Write a thorough analytical report covering all key metrics, trends, and "
                    f"notable findings. Be comprehensive and leave nothing out.")
        a_user = (f"Here is the complete dataset:\n{problem['dataset']}\n\n"
                  f"Please provide a comprehensive analytical report covering all metrics.")
    elif round_num == 2:
        a_system = ("You are a data analyst. You previously sent a comprehensive report to a specialist, "
                    "but they couldn't find some data they needed. Based on their feedback, "
                    "send a more targeted message. Focus on the specific metrics they seem to need.")
        a_user = (f"Dataset:\n{problem['dataset']}\n\n"
                  f"Your previous report was too verbose. The specialist's response was:\n"
                  f"\"{prev_b_content}\"\n\nSend a more focused summary targeting what they actually need.")
    else:  # round 3
        a_system = ("You are a data analyst. After two rounds, you now have a better sense of what the "
                    "specialist needs. Send only the essential data points in a concise format. "
                    "Be as brief as possible while ensuring the specialist has what they need.")
        a_user = (f"Dataset:\n{problem['dataset']}\n\n"
                  f"The specialist previously responded:\n"
                  f"\"{prev_b_content}\"\n\nSend only the essential data they need, as concisely as possible.")

    a_result = call_gpt(a_system, a_user)

    b_system = ("You are a specialist calculator. Using ONLY the information provided below, "
                "compute the requested value. If the exact data needed is not explicitly stated, "
                "output -999. Show your calculation and give the final numeric answer.")
    b_user = (f"DATA FROM ANALYST:\n{a_result['content']}\n\nTASK: {problem['b_task']}\n\n"
              f"Compute the answer using ONLY the data provided. If needed data is missing, output -999.")
    b_result = call_gpt(b_system, b_user)

    extracted = extract_number(b_result["content"])
    correct = check_accuracy(extracted, problem["answer"], problem["tolerance"])
    return {"a_tokens": a_result["tokens"], "b_answer": extracted, "correct": correct,
            "a_content": a_result["content"], "b_content": b_result["content"]}

print("Running Condition 4: Progressive (3 rounds)...")
results_pr = {"r1": [], "r2": [], "r3": []}

# Round 1
print("  Round 1...")
for p in PROBLEMS:
    print(f"    [PR1] Problem {p['id']}: {p['domain']}")
    result = run_progressive_round(p, 1, None)
    results_pr["r1"].append(result)
    time.sleep(1)

# Round 2 (depends on Round 1 results)
print("  Round 2...")
for i, p in enumerate(PROBLEMS):
    print(f"    [PR2] Problem {p['id']}: {p['domain']}")
    result = run_progressive_round(p, 2, results_pr["r1"][i]["b_content"])
    results_pr["r2"].append(result)
    time.sleep(1)

# Round 3 (depends on Round 2 results)
print("  Round 3...")
for i, p in enumerate(PROBLEMS):
    print(f"    [PR3] Problem {p['id']}: {p['domain']}")
    result = run_progressive_round(p, 3, results_pr["r2"][i]["b_content"])
    results_pr["r3"].append(result)
    time.sleep(1)

for rnd in ["r1", "r2", "r3"]:
    acc = sum(1 for r in results_pr[rnd] if r["correct"]) / len(results_pr[rnd]) * 100
    avg_tok = sum(r["a_tokens"] for r in results_pr[rnd]) / len(results_pr[rnd])
    print(f"  Progressive {rnd.upper()}: Avg A tokens = {avg_tok:.0f}, Accuracy = {acc:.1f}%")

## 결과 요약

In [ ]:
# ============================================================
# Results Summary: Table + Analysis + JSON Export
# ============================================================

def analyze(results, label):
    tokens = [r["a_tokens"] for r in results]
    accuracies = [1 if r["correct"] else 0 for r in results]
    return {
        "Condition": label,
        "Avg Tokens": round(sum(tokens) / len(tokens)),
        "Min Tokens": min(tokens),
        "Max Tokens": max(tokens),
        "Accuracy (%)": round(sum(accuracies) / len(accuracies) * 100, 1),
        "Correct": f"{sum(accuracies)}/{len(results)}",
    }

summary_data = [
    analyze(results_nc, "No Context"),
    analyze(results_ow, "One-Way"),
    analyze(results_mu, "Mutual"),
    analyze(results_pr["r1"], "Progressive R1"),
    analyze(results_pr["r2"], "Progressive R2"),
    analyze(results_pr["r3"], "Progressive R3"),
]

df_summary = pd.DataFrame(summary_data)
print("=" * 80)
print("                    KI-1 RESULTS SUMMARY")
print("=" * 80)
display(df_summary)

# Token reduction analysis
nc_tok = summary_data[0]["Avg Tokens"]
mu_tok = summary_data[2]["Avg Tokens"]
ow_tok = summary_data[1]["Avg Tokens"]
print(f"\n--- Token Reduction ---")
print(f"No Context -> Mutual:  {nc_tok} -> {mu_tok} tokens ({round((1 - mu_tok/nc_tok)*100)}% reduction)")
print(f"No Context -> One-Way: {nc_tok} -> {ow_tok} tokens ({round((1 - ow_tok/nc_tok)*100)}% reduction)")

# Per-problem detail table
print("\n--- Per-Problem Detail ---")
per_problem = []
for i, p in enumerate(PROBLEMS):
    per_problem.append({
        "ID": p["id"],
        "Domain": p["domain"],
        "Expected": p["answer"],
        "NC Tokens": results_nc[i]["a_tokens"],
        "OW Tokens": results_ow[i]["a_tokens"],
        "MU Tokens": results_mu[i]["a_tokens"],
        "NC Acc": "OK" if results_nc[i]["correct"] else "FAIL",
        "OW Acc": "OK" if results_ow[i]["correct"] else "FAIL",
        "MU Acc": "OK" if results_mu[i]["correct"] else "FAIL",
    })

df_detail = pd.DataFrame(per_problem)
display(df_detail)

# Save results to JSON
output = {
    "experiment": "KI-1 Natural Token",
    "model": MODEL,
    "temperature": TEMPERATURE,
    "timestamp": datetime.now().isoformat(),
    "design": "No max_tokens limit. Large datasets (15-20 points). B needs 2-5 specific values.",
    "summary": summary_data,
    "per_problem": per_problem,
}

out_path = "ki1_natural_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"\nResults saved to {out_path}")